# 04 — Async pipeline: `await arun`, `arun_many`, `process_batch_async`

v0.5 (PR #45) adds the async surface: `AsyncPipeline.arun` /
`AsyncPipeline.arun_many`, the bounded-concurrency helper
`process_batch_async`, and a default `Backend.acomplete` that backends can
override for native async HTTP.

**No API key required** — uses `MockBackend`. We subclass it with a tiny
`time.sleep` so the concurrency comparison below is observable in wall time.


In [ ]:
import asyncio
import logging
import tempfile
import time
from pathlib import Path

logging.getLogger("idp").setLevel(logging.CRITICAL)

from idp.batch import BatchItemResult, process_batch_async
from idp.llm.backend import CompletionRequest, Message, get_backend
from idp.llm.backend import MockBackend
from idp.pipeline.async_pipeline import AsyncPipeline

# Resolve a repo-relative scratch dir so cells work from any CWD
import os
_REPO_ROOT = Path(os.environ.get("IDP_REPO_ROOT", Path.cwd()))

## 1. Setup

The pipeline (and its backend) stay loaded across calls — building it once
and reusing it is the whole point of the batch helper. For the demo we'll
wrap `MockBackend` with a 50ms sleep so the concurrency comparison below
produces a measurable wall-time difference.


In [ ]:
class _SlowMock(MockBackend):
    """MockBackend + a tiny sleep so async concurrency is visible."""
    def complete(self, req):  # type: ignore[override]
        time.sleep(0.05)
        return super().complete(req)

backend = _SlowMock()
apipe = AsyncPipeline.from_backend(backend=backend, schema="Invoice")
print(f"backend: {apipe.pipeline.backend_name}")
print(f"schema:  {apipe.pipeline.schema.__name__}")

In [ ]:
# Write one fake invoice to a temp file so AsyncPipeline.arun() can
# resolve it via Document.from_path (arun accepts str | Path | Document).
TMP = Path(tempfile.mkdtemp(prefix="idp-async-"))
demo_text = "Invoice INV-001 from ACME for $100.00"
ONE = TMP / "inv-001.txt"
ONE.write_text(demo_text)

t = time.perf_counter()
result = await apipe.arun(ONE)
ms = (time.perf_counter() - t) * 1000
print(f"single-doc await: {ms:.0f}ms")
print(f"schema:    {result.schema_name}")
print(f"classify:  {result.classification}")

## 2. `arun_many` — concurrent extractions, input order preserved

`AsyncPipeline.arun_many(docs)` is a thin wrapper over
`asyncio.gather([self.arun(d) for d in docs])`. Results come back in the
same order as inputs, regardless of which one finishes first.


In [ ]:
# Five fake invoices, different amounts so we can see all five ran.
texts = [
    f"Invoice INV-{i:03d} from ACME for ${100 + i}.00"
    for i in range(1, 6)
]
many = []
for i, t in enumerate(texts, start=1):
    p = TMP / f"inv-{i:03d}.txt"
    p.write_text(t)
    many.append(p)

t = time.perf_counter()
results = await apipe.arun_many(many)
elapsed_ms = (time.perf_counter() - t) * 1000

print(f"docs:       {len(results)}")
print(f"wall time:  {elapsed_ms:.0f}ms (≈ 1 doc latency, run in parallel)")
print(f"order kept: {[Path(r.document.source_path).name for r in results]}")

## 3. `process_batch_async` — bounded concurrency helper

`process_batch_async` is the async analog of the sync `process_batch()`
covered in notebook 03. The `concurrency=N` parameter caps the number of
in-flight pipeline runs with an `asyncio.Semaphore`. Errors are captured
per-item in `BatchItemResult` — the helper never raises.

Two timing comparisons below make the concurrency behavior visible:
- `concurrency=1` → runs sequentially (≈ N × single-doc latency)
- `concurrency=5` → all five run in parallel (≈ single-doc latency)


In [ ]:
t = time.perf_counter()
res_serial = await process_batch_async(
    many, backend=backend, schema="Invoice", concurrency=1,
)
serial_ms = (time.perf_counter() - t) * 1000
single_ms = serial_ms / len(res_serial)
print(f"concurrency=1: {serial_ms:.0f}ms total, {single_ms:.0f}ms per doc")
print(f"  ok: {sum(1 for r in res_serial if r.ok)}/{len(res_serial)}")

In [ ]:
t = time.perf_counter()
res_par = await process_batch_async(
    many, backend=backend, schema="Invoice", concurrency=5,
)
par_ms = (time.perf_counter() - t) * 1000
print(f"concurrency=5: {par_ms:.0f}ms total")
print(f"  ok: {sum(1 for r in res_par if r.ok)}/{len(res_par)}")
speedup = serial_ms / par_ms if par_ms else float("inf")
print(f"  speedup vs serial: {speedup:.1f}x")

## 4. Progress callback

`on_progress(done, total)` fires after each item completes. Use it for
Databricks widget updates, log lines, or a tqdm progress bar.


In [ ]:
lines: list[str] = []

def _on_progress(done: int, total: int) -> None:
    lines.append(f"  progress: {done}/{total}")

await process_batch_async(
    many, backend=backend, schema="Invoice",
    concurrency=5, on_progress=_on_progress,
)
# Print in the order they fired
for line in lines:
    print(line)

## 5. Error handling

The helper never raises. Failures land in `BatchItemResult.error` and the
`.success` / `.latency_s` fields give you a clean per-item view. A missing
file is the easiest reproducer.


In [ ]:
mixed = many + [TMP / "does-not-exist.txt"]
results = await process_batch_async(
    mixed, backend=backend, schema="Invoice", concurrency=2,
)

print(f"{'path':<28} {'ok':<5} {'latency_s':<10} {'error':<30}")
print("-" * 75)
for r in results:
    err = (r.error or "")[:28]
    print(f"{Path(r.path).name:<28} {str(r.success):<5} {r.latency_s:<10.3f} {err:<30}")

ok_count = sum(1 for r in results if r.success)
err_count = sum(1 for r in results if not r.success)
print()
print(f"summary: {ok_count} ok, {err_count} failed (helper did NOT raise)")

## 6. `Backend.acomplete` — direct async surface

Every `Backend` exposes an async `acomplete(req)` method. The default
implementation runs the sync `complete()` in a worker thread via
`asyncio.to_thread`, so any existing backend transparently supports
`await`. Backends with a native async HTTP client (anthropic, openai)
override `acomplete` for better performance.


In [ ]:
req = CompletionRequest(
    messages=[Message(role="user", content="Say hi in JSON.")],
    json_mode=True,
)
t = time.perf_counter()
out = await backend.acomplete(req)
ms = (time.perf_counter() - t) * 1000
print(f"acomplete (default → asyncio.to_thread): {ms:.0f}ms")
print(f"output: {out!r}")

## When to use sync vs async

| context | use |
|---|---|
| Plain Python script, one doc | `Pipeline.run` (sync) |
| Plain Python script, many docs | `process_batch` (sync generator) |
| FastAPI / aiohttp handler | `await AsyncPipeline.arun` |
| Jupyter with `ipykernel` | `await AsyncPipeline.arun` |
| Many docs in an async context | `await process_batch_async(...)` |
| You wrote your own HTTP client | `await backend.acomplete(req)` |

## What's next

- **Notebook 01** — single-doc pipeline, six stages explained.
- **Notebook 02** — HITL feedback loop, low-confidence fields, policy override.
- **Notebook 03** — sync `process_batch()` with checkpoint / resume.
